# RAS: better representations, local heads, and compilation gaps

Compare MiniLM, DenseOn, pooled LateOn and learned LateOn token-max predicates on explicit WANDS attributes. Each has global/local heads and FP32 → Binary1 → Binary1+int4 controls.

**Default: 12,000 products, one seed, K=8/16 local experts.** Select a GPU runtime. The first redacted encoding pass is cached; subsequent head experiments reuse it. This notebook does not rerun full-corpus retrieval or MUVERA.

Labels are catalog metadata, not CLIP or independent aesthetic judgments. Titles/descriptions omit the structured label fields. Recall is conditional on held-out products with known labels for every term. [Full protocol](https://github.com/hanialshater/ras/blob/codex/colbert-muvera-baselines/docs/WANDS_PREDICATES.md).


In [ ]:
import os, sys, subprocess, json, hashlib, shutil, time
from pathlib import Path
from collections import deque
from google.colab import drive
drive.mount('/content/drive')
os.environ.update(USE_TF='0', USE_FLAX='0', PYLATE_SCORES_BACKEND='torch',
                  OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
REPO = Path('/content/ras-predicates')
BRANCH = 'codex/colbert-muvera-baselines'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/hanialshater/ras.git', str(REPO)], check=True)
else:
    branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO, text=True).strip()
    assert branch == BRANCH, f'Unexpected checkout: {branch}'
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[wands,wands-systems,dev]'], cwd=REPO, check=True)
REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('Repository revision:', REVISION)


## Settings and resumable storage

Keep the embedding cache when changing heads. Change the settings below to create a separate study output automatically. Use `CLUSTERS = []` for a global-only first pass. Keep the small pilot before trying all 42,994 products (`MAX_PRODUCTS = 0`) or more seeds.


In [ ]:
MAX_PRODUCTS = 12000
VIEW = 'redacted'
CLUSTERS = [8, 16]
SEEDS = [7]
EPOCHS = 3
POOL_SIZE, K = 500, 50
MODELS = ['minilm', 'dense', 'colbert']
ENCODE_BATCH, HEAD_BATCH = 16, 64
MAX_LENGTH = 512

STORE = Path('/content/drive/MyDrive/ras_wands_predicates')
LOCAL = Path('/content/ras_predicate_cache')
LOGS = STORE / 'logs'
for folder in [STORE, LOCAL, LOGS]: folder.mkdir(parents=True, exist_ok=True)
SETTINGS = dict(view=VIEW, max_products=MAX_PRODUCTS, clusters=CLUSTERS, seeds=SEEDS,
                epochs=EPOCHS, pool_size=POOL_SIZE, k=K, models=MODELS, max_length=MAX_LENGTH,
                encode_batch=ENCODE_BATCH, head_batch=HEAD_BATCH, revision=REVISION)
TAG = hashlib.sha256(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
DATA_REL = Path('data') / f'v1-{VIEW}-{MAX_PRODUCTS}'
EMB_REL = Path('embeddings')
RUN_REL = Path('studies') / TAG
DATA, EMB, RUN = [LOCAL / p for p in [DATA_REL, EMB_REL, RUN_REL]]

def copy_tree(source, destination):
    if not source.exists(): return
    copied = 0
    for path in source.rglob('*'):
        if not path.is_file() or '.tmp' in path.name: continue
        target = destination / path.relative_to(source)
        if target.exists() and path.stat().st_size == target.stat().st_size and abs(path.stat().st_mtime - target.stat().st_mtime) < .01:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_name(target.name + '.tmp')
        shutil.copy2(path, temp)
        temp.replace(target)
        copied += 1
    print(f'Checkpoint: {copied} files copied to {destination}', flush=True)

for rel in [DATA_REL, EMB_REL, RUN_REL]: copy_tree(STORE / rel, LOCAL / rel)

def backup(*rels):
    for rel in rels: copy_tree(LOCAL / rel, STORE / rel)

def run_logged(command, name):
    log = LOGS / f'{TAG}-{name}.log'
    tail = deque(maxlen=40)
    print('Python:', sys.version.split()[0], '| executable:', sys.executable, '| Log:', log, flush=True)
    with log.open('w') as out:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); out.write(line); out.flush(); tail.append(line)
            status = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if status: raise RuntimeError(f'Exit {status}. Log: {log}\n' + ''.join(tail))

BASE = [sys.executable, '-u', '-m', 'experiments.wands_predicates',
        '--data-dir', str(DATA), '--embedding-dir', str(EMB), '--output-dir', str(RUN),
        '--view', VIEW, '--max-products', str(MAX_PRODUCTS), '--device', 'cuda',
        '--max-length', str(MAX_LENGTH), '--encode-batch', str(ENCODE_BATCH),
        '--head-batch', str(HEAD_BATCH), '--epochs', str(EPOCHS), '--pool-size', str(POOL_SIZE),
        '--k', str(K), '--clusters', *map(str, CLUSTERS), '--seeds', *map(str, SEEDS)]
print('Study output:', STORE / RUN_REL)


## 1. Check the environment and audit labels

Read the positive/negative examples. Missing and conflicting attributes stay unknown. There is no model download during the label audit.


In [ ]:
run_logged([sys.executable, '-c', 'import torch, pylate, sentence_transformers; assert torch.cuda.is_available(), "Select a GPU runtime"; print(torch.cuda.get_device_name(0))'], 'environment')
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_wands_predicates.py'], 'tests')
try:
    run_logged(BASE + ['--phase', 'audit'], 'audit')
finally:
    backup(DATA_REL)
import pandas as pd
from IPython.display import display
display(pd.read_csv(DATA / 'label_audit.csv'))
display(pd.DataFrame(json.loads((DATA / 'label_examples.json').read_text())))
print((DATA / 'scope.json').read_text())


## 2. Encode once

This is the longer first step. Titles/descriptions require their own cache because the earlier full-text embeddings contained the target metadata. Each completed model is backed up separately. An interrupted run resumes saved 512-product shards. Model revisions, text and package versions identify the cache; head settings do not invalidate it.


In [ ]:
for model in MODELS:
    try:
        run_logged(BASE + ['--phase', 'encode', '--models', model], 'encode-' + model)
    finally:
        backup(EMB_REL)
print('All requested embeddings ready.')


## 3. Fit and compile the heads

Training/calibration/test product groups are separate. KMeans and centroids fit only on training products. Sparse local heads use explicit global fallback. Every precision arm ranks the same candidates and gets calibration from the same calibration partition. Stored raw scores and heads are reused on an identical rerun.

Token-max is a learned attribute head over frozen LateOn tokens. The comparison includes pooled LateOn to test whether token structure helps. Binary1 remains one code **per token** in that arm.


In [ ]:
RUN.mkdir(parents=True, exist_ok=True)
(RUN / 'notebook_settings.json').write_text(json.dumps(SETTINGS, indent=2))
try:
    run_logged(BASE + ['--phase', 'study', '--models', *MODELS], 'study')
finally:
    backup(RUN_REL)


## 4. Read the gap decomposition

For each composition, the signed losses sum to **1 − Binary1+int4 recall**:

1. Relevant items outside the candidate pool.
2. Relevant candidates beyond the return budget K.
3. Oracle → FP32 modelling loss.
4. FP32 → Binary1 item-compilation loss.
5. Binary1 → int4 weight-compilation loss.

This is agreement with **catalog attribute labels**, not agreement with exact LateOn. Recall denominators use the full annotated test universe for that plan. Zero-positive plans have undefined recall and remain counted. AP/F1 summarize individual predicates; composition recall summarizes singles and train-selected conjunctions/negations. A one-seed pilot is not a generalization guarantee.


In [ ]:
summary = pd.read_csv(RUN / 'summary.csv')
gaps = pd.read_csv(RUN / 'gap_summary.csv')
payloads = pd.read_csv(RUN / 'payloads.csv')
print('Predicate and composition quality; dense_query is the pretrained retrieval control')
display(summary)
print('Coverage, return-budget, FP32 and compilation gaps')
display(gaps)
print('Representation and predicate-program payloads; not process memory or serving latency')
display(payloads[['method', 'seed', 'item_payload_bytes_mean', 'program_payload_bytes',
                  'shared_encoder_bytes', 'routing_centers_bytes', 'fallback_heads', 'head_count']])
for seed in SEEDS:
    print('Split:', (RUN / f'seed_{seed}' / 'split_audit.json').read_text())
print('Saved outputs:', STORE / RUN_REL)


Do not compare the token head's storage to a single-vector sidecar without including all tokens and offsets. This evaluator reconstructs packed values for numerical quality; it makes no optimized Binary1 serving-speed claim. Use these results to choose a representation/head before an isolated latency, memory and ANN traversal experiment.
